# 01_exploration_francetravail

Les URL (FRANCE TRAVAIL) utiles sont :

- https://francetravail.io/data/api/offres-emploi
- https://francetravail.io/data/api/offres-emploi/documentation#/

In [1]:
import os
from dotenv import load_dotenv
import requests
import logging
import time
import json

In [2]:
# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)

# Charge le .env en mémoire
load_dotenv()

True

In [3]:
##################  VARIABLES  ##################
# France Travail
FRANCETRAVAIL_CLIENT_ID     = os.getenv("FRANCETRAVAIL_CLIENT_ID")
FRANCETRAVAIL_CLIENT_SECRET = os.getenv("FRANCETRAVAIL_CLIENT_SECRET")

In [4]:
# Vérification de l'accès aux variables relatives à la connexion à l'API de France Travail
variables = [
    "FRANCETRAVAIL_CLIENT_ID",
    "FRANCETRAVAIL_CLIENT_SECRET",
]

for var in variables:
    valeur = os.getenv(var)
    print(f"{var} : {'✅ chargée' if valeur else '❌ manquante'}")

FRANCETRAVAIL_CLIENT_ID : ✅ chargée
FRANCETRAVAIL_CLIENT_SECRET : ✅ chargée


In [5]:
# ---------------------------
# AUTH FRANCE TRAVAIL
# ---------------------------

def get_token(retries=3, wait=5):
    """Récupère un token d'authentification OAuth2."""
    url = "https://entreprise.francetravail.fr/connexion/oauth2/access_token"
    params = {"realm": "/partenaire"}
    data = {
        "grant_type":    "client_credentials",
        "client_id":     FRANCETRAVAIL_CLIENT_ID,
        "client_secret": FRANCETRAVAIL_CLIENT_SECRET,
        "scope":         "api_offresdemploiv2 o2dsoffre"
    }
    for attempt in range(retries):
        try:
            response = requests.post(url, params=params, data=data)
            return response.json()["access_token"]
        except requests.RequestException as e:
                logging.warning(f"Erreur OAuth attempt {attempt+1}: {e}")
                time.sleep(wait)
    raise RuntimeError("Impossible d'obtenir un token OAuth après plusieurs essais.")

In [6]:
# ---------------------------
# REQUETE API FRANCE TRAVAIL
# ---------------------------
def rechercher_offres(token, mots_cles="data engineer", nb_resultats=100):
    """Recherche des offres d'emploi via l'API FranceTravail."""
    url = "https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search"
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept":        "application/json"
    }
    params = {
        "motsCles":   mots_cles,
        "range":      f"0-{nb_resultats - 1}",
        "sort":       "1"
    }

    try: 
        response = requests.get(url, headers=headers, params=params)
        return response.json()
    except requests.RequestException as e:
        print(f"Erreur API France Travail: {e}")

In [7]:
if __name__ == "__main__":
    
    # Récupération token France travail
    token  = get_token()

    # Requête vers API france Travail
    offres = rechercher_offres(token, "data engineer", nb_resultats = 3)
    print(f"{len(offres.get('resultats', []))} offres récupérées") 

3 offres récupérées


In [11]:
offres['resultats'][0]

{'id': '1182550',
 'intitule': 'Technical data wiring engineer (f/h)',
 'description': "<b><b>Job Description:</b></b> <br><br> <b>Airbus Commercial Aircraft</b> recherche un <b>Technical Data Wiring Engineer (f/h)</b> pour rejoindre notre département Technical Data basé à <b>Toulouse, France.</b> <br><br>Vous ferez partie d'une équipe qui développe, livre et supporte la documentation de maintenance aéronautique. Au sein de l'équipe Schematics & Wiring, vous serez impliqué(e) dans la gestion des données techniques électriques (AWM, ASM, AWL) visant à garantir la navigabilité et la sécurité des opérations de maintenance pour nos clients mondiaux. <br><br> <b>Votre environnement de travail :</b> <br><br>Capitale mondiale de l'aéronautique et capitale européenne de la recherche dans le spatial, Toulouse est une ville dynamique du sud-ouest de la France desservie par un aéroport international. Idéalement située entre la mer Méditerranée et l'océan Atlantique et à proximité des Pyrénées, el